### Vertical-velocity workflow:
    ds       = load_model(run_dir, iters)
    uv       = sample_fields(ds, positions, vars=('UVEL', 'VVEL'))
    w_est    = compute_w_planefit(uv)['w_est']
    w_model  = sample_model_w(ds, positions)
    fig      = plot_w_comparison(w_est, w_model)

In [ ]:
# load python packages
import json
import xarray as xr
from osse_tools import sample_fields, compute_w_planefit, load_model
import warnings
# suppress warnings for cleaner output
warnings.filterwarnings("ignore")

In [11]:
# load glider positions from config file, in this example it is an array of 6 gliders around a single mooring at 0N, 140W (7 positions total)

CONFIG = "/home/edavenport/analysis/tpose24-osse/experiments/experiment_1/configs/equator/equator_hex1deg_w0.5.json"
# get positions from json config
with open(CONFIG, 'r') as f:
    config = json.load(f)
all_positions = config['positions']
all_positions

[[0.5, 220.0],
 [-0.5, 220.0],
 [0.25, 219.5],
 [0.25, 220.5],
 [-0.25, 219.5],
 [-0.25, 220.5],
 [0.0, 220.0]]

In [9]:
# load model output 

RUN_DIR    = '/data/SO3/edavenport/tpose24/oct2012_3month_transp_cons'
ITERS      = list(range(36, 26173, 36))     # 3-hourly diag_state steps (this is model-specific)
MIN_DEPTH  = 8                              # shallowest sampled depth (m)
MAX_DEPTH  = 80
DZ_OBS     = 2

# open dataset -- this load_model function is model-specific, time step is 300s.
# ds is the xarray dataset containing the model output, which can be used for sampling and analysis.
ds = load_model(RUN_DIR, ITERS)
ds

<xarray.Dataset> Size: 553GB
Dimensions:  (time: 727, Z: 138, YC: 384, XC: 512, XG: 512, YG: 384, Zl: 138,
              Zp1: 139, Zu: 138)
Coordinates: (12/43)
  * time     (time) datetime64[ns] 6kB 2012-10-01T03:00:00 ... 2012-12-30T21:...
  * Z        (Z) float64 1kB -0.5 -1.5 -2.5 ... -5.25e+03 -5.35e+03 -5.45e+03
  * YC       (YC) float64 3kB -5.479 -5.438 -5.396 -5.354 ... 10.4 10.44 10.48
  * XC       (XC) float64 4kB 209.4 209.4 209.4 209.5 ... 230.6 230.6 230.6
  * XG       (XG) float64 4kB 209.3 209.4 209.4 209.5 ... 230.5 230.6 230.6
  * YG       (YG) float64 3kB -5.5 -5.458 -5.417 -5.375 ... 10.38 10.42 10.46
    ...       ...
    rSurfC   (YC, XC) >f4 786kB dask.array<chunksize=(384, 512), meta=np.ndarray>
    rLowW    (YC, XG) >f4 786kB dask.array<chunksize=(384, 512), meta=np.ndarray>
    rLowS    (YG, XC) >f4 786kB dask.array<chunksize=(384, 512), meta=np.ndarray>
    rSurfS   (YG, XC) >f4 786kB dask.array<chunksize=(384, 512), meta=np.ndarray>
    maskInS  (YG, XC) bool 197kB dask.array<chunksize=(384, 512), meta=np.ndarray>
    iter     (time) int64 6kB dask.array<chunksize=(1,), meta=np.ndarray>
Data variables:
    THETA    (time, Z, YC, XC) float32 79GB dask.array<chunksize=(1, 138, 384, 512), meta=np.ndarray>
    SALT     (time, Z, YC, XC) float32 79GB dask.array<chunksize=(1, 138, 384, 512), meta=np.ndarray>
    UVEL     (time, Z, YC, XG) float32 79GB dask.array<chunksize=(1, 138, 384, 512), meta=np.ndarray>
    VVEL     (time, Z, YG, XC) float32 79GB dask.array<chunksize=(1, 138, 384, 512), meta=np.ndarray>
    WVEL     (time, Zl, YC, XC) float32 79GB dask.array<chunksize=(1, 138, 384, 512), meta=np.ndarray>
    PHIHYD   (time, Z, YC, XC) float32 79GB dask.array<chunksize=(1, 138, 384, 512), meta=np.ndarray>
    DRHODR   (time, Zl, YC, XC) float32 79GB dask.array<chunksize=(1, 138, 384, 512), meta=np.ndarray>
Attributes:
    Conventions:  CF-1.6
    title:        netCDF wrapper of MITgcm MDS binary data
    source:       MITgcm
    history:      Created by calling `open_mdsdataset(data_dir='/data/SO3/eda...

In [12]:
uv = sample_fields(ds, all_positions, vars=('UVEL', 'VVEL'), max_depth=MAX_DEPTH, dz_obs=DZ_OBS, min_depth=MIN_DEPTH).compute()
uv

<xarray.Dataset> Size: 3MB
Dimensions:    (glider: 7, obs_depth: 36, time: 727)
Coordinates: (12/26)
  * glider     (glider) int64 56B 0 1 2 3 4 5 6
  * obs_depth  (obs_depth) float64 288B -9.0 -11.0 -13.0 ... -75.0 -77.0 -79.0
  * time       (time) datetime64[ns] 6kB 2012-10-01T03:00:00 ... 2012-12-30T2...
    dyG        (glider) float64 56B 4.632e+03 4.632e+03 ... 4.632e+03 4.632e+03
    dxC        (glider) float64 56B 4.632e+03 4.632e+03 ... 4.632e+03 4.632e+03
    rAw        (glider) float64 56B 2.146e+07 2.146e+07 ... 2.146e+07 2.146e+07
    ...         ...
    rLowS      (glider) float64 56B -4.294e+03 -4.295e+03 ... -4.333e+03
    rSurfS     (glider) float64 56B 0.0 0.0 0.0 0.0 0.0 0.0 0.0
    XC         (glider) float64 56B 220.0 220.0 219.5 220.5 219.5 220.5 220.0
    YG         (glider) float64 56B 0.5 -0.5 0.25 0.25 -0.25 -0.25 0.0
    lat        (glider) float64 56B 0.5 -0.5 0.25 0.25 -0.25 -0.25 0.0
    lon        (glider) float64 56B 220.0 220.0 219.5 220.5 219.5 220.5 220.0
Data variables:
    U          (time, glider, obs_depth) float64 1MB -0.2903 -0.2802 ... 0.122
    V          (time, glider, obs_depth) float64 1MB 0.2351 0.2325 ... -0.1083

In [15]:
uv.U

<xarray.DataArray 'U' (time: 727, glider: 7, obs_depth: 36)> Size: 1MB
array([[[-0.29033717, -0.28018009, -0.27086148, ...,  0.41954967,
          0.47253516,  0.52614609],
        [-0.21510199, -0.20449927, -0.19454114, ...,  0.48935171,
          0.5316403 ,  0.57546522],
        [-0.22175112, -0.21105695, -0.20102397, ...,  0.52955   ,
          0.58072525,  0.63340769],
        ...,
        [-0.1281554 , -0.11704628, -0.10662137, ...,  0.57119965,
          0.61311706,  0.6560921 ],
        [-0.21823198, -0.20710971, -0.19655541, ...,  0.56201572,
          0.60460072,  0.64821452],
        [-0.19425541, -0.18290608, -0.17216736, ...,  0.54280062,
          0.58562555,  0.62962371]],

       [[-0.28405761, -0.27931749, -0.27486925, ...,  0.40474334,
          0.45718578,  0.51088823],
        [-0.16722674, -0.16208228, -0.1571481 , ...,  0.49514504,
          0.53633055,  0.58016762],
        [-0.17295717, -0.1676055 , -0.16253565, ...,  0.52821118,
          0.57919158,  0.63148198],
...
        [-0.49673987, -0.49250351, -0.48885317, ...,  0.01313626,
          0.04061922,  0.067863  ],
        [-0.5360682 , -0.53210761, -0.52871553, ..., -0.04419084,
         -0.00989858,  0.02446254],
        [-0.56029387, -0.55616733, -0.55256822, ...,  0.04978615,
          0.08455635,  0.11905908]],

       [[-0.82246218, -0.81165098, -0.80246307, ...,  0.00387923,
          0.02445031,  0.04591222],
        [-0.5516595 , -0.54100853, -0.53253154, ..., -0.05827898,
         -0.02837651,  0.0012673 ],
        [-0.61836382, -0.6087206 , -0.60100021, ...,  0.0318079 ,
          0.07348545,  0.11412816],
        ...,
        [-0.50573904, -0.49576378, -0.48786177, ...,  0.00146245,
          0.02867851,  0.05586733],
        [-0.5582613 , -0.54848167, -0.54066356, ..., -0.01792605,
          0.01566429,  0.04959154],
        [-0.58480849, -0.57510352, -0.56728161, ...,  0.05246804,
          0.08733897,  0.12197339]]], shape=(727, 7, 36))
Coordinates: (12/26)
  * time       (time) datetime64[ns] 6kB 2012-10-01T03:00:00 ... 2012-12-30T2...
  * glider     (glider) int64 56B 0 1 2 3 4 5 6
  * obs_depth  (obs_depth) float64 288B -9.0 -11.0 -13.0 ... -75.0 -77.0 -79.0
    dyG        (glider) float64 56B 4.632e+03 4.632e+03 ... 4.632e+03 4.632e+03
    dxC        (glider) float64 56B 4.632e+03 4.632e+03 ... 4.632e+03 4.632e+03
    rAw        (glider) float64 56B 2.146e+07 2.146e+07 ... 2.146e+07 2.146e+07
    ...         ...
    rLowS      (glider) float64 56B -4.294e+03 -4.295e+03 ... -4.333e+03
    rSurfS     (glider) float64 56B 0.0 0.0 0.0 0.0 0.0 0.0 0.0
    XC         (glider) float64 56B 220.0 220.0 219.5 220.5 219.5 220.5 220.0
    YG         (glider) float64 56B 0.5 -0.5 0.25 0.25 -0.25 -0.25 0.0
    lat        (glider) float64 56B 0.5 -0.5 0.25 0.25 -0.25 -0.25 0.0
    lon        (glider) float64 56B 220.0 220.0 219.5 220.5 219.5 220.5 220.0
Attributes:
    standard_name:  UVEL
    long_name:      Zonal Component of Velocity (m/s)
    units:          m/s
    mate:           VVEL

In [14]:
# save uv dataset to netcdf
uv.to_netcdf('/data/SO3/edavenport/tpose24/cache/uv_sampled_fields.nc')

In [18]:
# compute plane fit from sampled fields 
ds_est = compute_w_planefit(uv)

In [19]:
ds_est

<xarray.Dataset> Size: 478kB
Dimensions:    (time: 727, depth: 41, obs_depth: 40)
Coordinates:
  * time       (time) datetime64[ns] 6kB 2012-10-01T03:00:00 ... 2012-12-30T2...
  * depth      (depth) float64 328B 0.0 -2.0 -4.0 -6.0 ... -76.0 -78.0 -80.0
  * obs_depth  (obs_depth) float64 320B -1.0 -3.0 -5.0 ... -75.0 -77.0 -79.0
Data variables:
    w_est      (time, depth) float64 238kB 0.0 1.252e-06 ... 3.558e-06 4.515e-06
    div        (time, obs_depth) float64 233kB 6.262e-07 6.175e-07 ... 4.781e-07

In [20]:
ds_est.to_netcdf('/data/SO3/edavenport/tpose24/cache/w_estimated_fields.nc')